In [1]:
# Install transformers
#!pip install transformers

## Google Form Answers

In [2]:
# Import the necessary libraries
import os
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from transformers import AutoFeatureExtractor, ASTForAudioClassification

c:\Users\Atul\anaconda3\envs\gpu_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
GENRES = ["blues", "classical", "country", "disco", "hiphop",
          "jazz", "metal", "pop", "reggae", "rock"]
SR = 16000
DURATION = 10

In [4]:
# --- Q1: Validation set size ---
recipes = []
for g in GENRES:
    for i in range(100):
        recipes.append({"genre": g, "index": i})

train_recipes, val_recipes = train_test_split(
    recipes, test_size=0.2, shuffle=True, random_state=42
)
print(f"Q1 — Validation set size: {len(val_recipes)}")

# --- Q2: Mix array shape ---
target_length = SR * DURATION
mix = np.zeros(target_length, dtype=np.float32)
print(f"Q2 — Mix array shape: {mix.shape}")

# --- Q3: Feature extractor output shape ---
feature_extractor = AutoFeatureExtractor.from_pretrained(
    "MIT/ast-finetuned-audioset-10-10-0.4593"
)
dummy_mix = np.ones(160000)
inputs = feature_extractor(dummy_mix, sampling_rate=16000, return_tensors="pt")
spec = inputs["input_values"].squeeze(0)
print(f"Q3 — Feature extractor output shape: {spec.shape}")

# --- Q4: Trainable parameters ---
model = ASTForAudioClassification.from_pretrained(
    "MIT/ast-finetuned-audioset-10-10-0.4593",
    num_labels=10,
    ignore_mismatched_sizes=True
)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Q4 — Trainable parameters: {total_params}")

# --- Q5: Normalization math ---
y_test = np.array([-0.85, 0.40, 0.20, -0.10])
y_norm = y_test / (np.max(np.abs(y_test)) + 1e-9)
print(f"Q5 — Normalized value at index 0: {y_norm[0]:.3f}")

Q1 — Validation set size: 200
Q2 — Mix array shape: (160000,)
Q3 — Feature extractor output shape: torch.Size([1024, 128])


Some weights of ASTForAudioClassification were not initialized from the model checkpoint at MIT/ast-finetuned-audioset-10-10-0.4593 and are newly initialized because the shapes did not match:
- classifier.dense.bias: found shape torch.Size([527]) in the checkpoint and torch.Size([10]) in the model instantiated
- classifier.dense.weight: found shape torch.Size([527, 768]) in the checkpoint and torch.Size([10, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Q4 — Trainable parameters: 86196490
Q5 — Normalized value at index 0: -1.000


## AST Fine-tuning and training 

In [5]:
# Import the necessary libraries
import os
import glob
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import librosa
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score
from transformers import AutoFeatureExtractor, ASTForAudioClassification
import wandb
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")


In [6]:
# --- Configuration ---
DATA_ROOT = '../data/messy_mashup/genres_stems'
NOISE_DIR = '../data/messy_mashup/ESC-50-master/audio'
MODEL_DIR = '../models'
CACHE_DIR = '../working/ast_features'
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

GENRES = ["blues", "classical", "country", "disco", "hiphop",
          "jazz", "metal", "pop", "reggae", "rock"]
GENRE_TO_IDX = {g: i for i, g in enumerate(GENRES)}
IDX_TO_GENRE = {i: g for g, i in GENRE_TO_IDX.items()}
STEMS = ['drums.wav', 'vocals.wav', 'bass.wav', 'other.wav']

SR = 16000
DURATION = 10
TARGET_LENGTH = SR * DURATION
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

Using device: cuda


In [7]:
# Reproducibility
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)


In [8]:
# --- Step 1: Build lazy-loading recipes ---
def build_recipes(stems_dir, noise_dir, recipes_per_genre=100):
    noise_files = glob.glob(os.path.join(noise_dir, '**', '*.wav'), recursive=True)
    recipes = []

    for genre in GENRES:
        genre_path = os.path.join(stems_dir, genre)
        song_folders = sorted([
            os.path.join(genre_path, d)
            for d in os.listdir(genre_path)
            if os.path.isdir(os.path.join(genre_path, d))
        ])

        for i in range(recipes_per_genre):
            chosen_songs = random.sample(song_folders, 4)
            stem_paths = []
            for song, stem_type in zip(chosen_songs, STEMS):
                stem_paths.append(os.path.join(song, stem_type))

            recipes.append({
                "genre": genre,
                "label": GENRE_TO_IDX[genre],
                "stem_paths": stem_paths,
                "noise_path": random.choice(noise_files),
                "noise_intensity": random.uniform(0.1, 0.4),
            })

    return recipes


recipes = build_recipes(DATA_ROOT, NOISE_DIR, recipes_per_genre=100)
train_recipes, val_recipes = train_test_split(
    recipes, test_size=0.2, shuffle=True, random_state=42
)
print(f"Train recipes: {len(train_recipes)}, Val recipes: {len(val_recipes)}")

Train recipes: 800, Val recipes: 200


In [9]:
# --- Step 2: Precompute features ---


feature_extractor = AutoFeatureExtractor.from_pretrained(
    "MIT/ast-finetuned-audioset-10-10-0.4593"
)


def pad_or_trim(y, target_length):
    if len(y) < target_length:
        y = np.pad(y, (0, target_length - len(y)))
    else:
        y = y[:target_length]
    return y


def mix_recipe(recipe):
    """Mix stems + noise for a single recipe, return normalized 1D array."""
    mix = np.zeros(TARGET_LENGTH, dtype=np.float32)
    for stem_path in recipe["stem_paths"]:
        try:
            y, _ = librosa.load(stem_path, sr=SR, duration=DURATION)
            y = pad_or_trim(y, TARGET_LENGTH)
            mix += y
        except Exception:
            pass

    try:
        noise, _ = librosa.load(recipe["noise_path"], sr=SR)
        noise = pad_or_trim(noise, TARGET_LENGTH)
        mix += noise * recipe["noise_intensity"]
    except Exception:
        pass

    peak = np.max(np.abs(mix))
    if peak > 0:
        mix = mix / (peak + 1e-9)
    return mix


def precompute_features(recipes, split_name):
    """Run feature extraction once and save tensors to disk."""
    split_dir = os.path.join(CACHE_DIR, split_name)
    os.makedirs(split_dir, exist_ok=True)

    # Skip if already precomputed
    existing = glob.glob(os.path.join(split_dir, '*.pt'))
    if len(existing) == len(recipes):
        print(f"{split_name}: {len(existing)} cached files found, skipping precomputation.")
        return

    print(f"Precomputing {len(recipes)} features for {split_name}...")
    for i, recipe in enumerate(tqdm(recipes, desc=f"Precompute [{split_name}]")):
        mix = mix_recipe(recipe)
        inputs = feature_extractor(mix, sampling_rate=SR, return_tensors="pt")
        input_values = inputs["input_values"].squeeze(0)  # (1024, 128)
        label = recipe["label"]

        torch.save({"features": input_values, "label": label},
                    os.path.join(split_dir, f"sample_{i:04d}.pt"))

    print(f"{split_name}: precomputation complete.")


precompute_features(train_recipes, "train")
precompute_features(val_recipes, "val")


Precomputing 800 features for train...


Precompute [train]: 100%|██████████| 800/800 [00:50<00:00, 15.86it/s]


train: precomputation complete.
Precomputing 200 features for val...


Precompute [val]: 100%|██████████| 200/200 [00:11<00:00, 17.38it/s]

val: precomputation complete.


In [10]:
# --- Step 3: Dataset that loads precomputed tensors ---
class CachedFeatureDataset(Dataset):
    def __init__(self, cache_dir):
        self.files = sorted(glob.glob(os.path.join(cache_dir, '*.pt')))

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        data = torch.load(self.files[idx])
        return data["features"], data["label"]


train_dataset = CachedFeatureDataset(os.path.join(CACHE_DIR, "train"))
val_dataset = CachedFeatureDataset(os.path.join(CACHE_DIR, "val"))
print(f"Train samples: {len(train_dataset)}, Val samples: {len(val_dataset)}")

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=0)

Train samples: 800, Val samples: 200


In [11]:
# Shape check
sample_input, sample_label = train_dataset[0]
print(f"Sample input shape: {sample_input.shape}")
print(f"Sample label: {sample_label} ({IDX_TO_GENRE[sample_label]})")

Sample input shape: torch.Size([1024, 128])
Sample label: 0 (blues)


In [16]:
# --- Step 4: Initialize AST model with frozen base ---
model = ASTForAudioClassification.from_pretrained(
    "MIT/ast-finetuned-audioset-10-10-0.4593",
    num_labels=10,
    ignore_mismatched_sizes=True
)

# Freeze the entire base transformer
for param in model.audio_spectrogram_transformer.parameters():
    param.requires_grad = False

model = model.to(DEVICE)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters (classifier head only): {total_params}")

Some weights of ASTForAudioClassification were not initialized from the model checkpoint at MIT/ast-finetuned-audioset-10-10-0.4593 and are newly initialized because the shapes did not match:
- classifier.dense.bias: found shape torch.Size([527]) in the checkpoint and torch.Size([10]) in the model instantiated
- classifier.dense.weight: found shape torch.Size([527, 768]) in the checkpoint and torch.Size([10, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Trainable parameters (classifier head only): 9226


In [17]:
# --- Step 5: Training loop ---
optimizer = optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.01)
criterion = nn.CrossEntropyLoss()
num_epochs = 10

wandb.init(
    project="21f3002298-dl-genai-project",
    name="ast-milestone5_10epochs",
    config={
        "model": "AST (MIT/ast-finetuned-audioset-10-10-0.4593)",
        "epochs": num_epochs,
        "batch_size": 32,
        "lr": 5e-5,
        "optimizer": "AdamW",
        "weight_decay": 0.01,
        "sr": SR,
        "duration": DURATION,
        "recipes_per_genre": 100,
        "frozen_base": True,
    }
)

best_val_f1 = 0.0

In [18]:
for epoch in range(num_epochs):
    # --- Training ---
    model.train()
    train_loss = 0.0
    train_preds, train_labels = [], []

    for batch_inputs, batch_labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]"):
        batch_inputs = batch_inputs.to(DEVICE)
        batch_labels = batch_labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(batch_inputs)
        loss = criterion(outputs.logits, batch_labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * batch_inputs.size(0)
        preds = outputs.logits.argmax(dim=1).cpu().numpy()
        train_preds.extend(preds)
        train_labels.extend(batch_labels.cpu().numpy())

    train_loss /= len(train_dataset)
    train_acc = accuracy_score(train_labels, train_preds)
    train_f1 = f1_score(train_labels, train_preds, average='macro')

    # --- Validation ---
    model.eval()
    val_loss = 0.0
    val_preds, val_labels = [], []

    with torch.no_grad():
        for batch_inputs, batch_labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Val]"):
            batch_inputs = batch_inputs.to(DEVICE)
            batch_labels = batch_labels.to(DEVICE)

            outputs = model(batch_inputs)
            loss = criterion(outputs.logits, batch_labels)

            val_loss += loss.item() * batch_inputs.size(0)
            preds = outputs.logits.argmax(dim=1).cpu().numpy()
            val_preds.extend(preds)
            val_labels.extend(batch_labels.cpu().numpy())

    val_loss /= len(val_dataset)
    val_acc = accuracy_score(val_labels, val_preds)
    val_f1 = f1_score(val_labels, val_preds, average='macro')

    wandb.log({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "train_f1": train_f1,
        "val_loss": val_loss,
        "val_acc": val_acc,
        "val_f1": val_f1,
    })

    print(f"Epoch {epoch+1}: train_loss={train_loss:.4f} train_f1={train_f1:.4f} | "
          f"val_loss={val_loss:.4f} val_f1={val_f1:.4f}")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), os.path.join(MODEL_DIR, 'ast_best_10epochs.pth'))
        print(f"  Saved best model (val_f1={val_f1:.4f})")

Epoch 1/10 [Val]: 100%|██████████| 7/7 [00:39<00:00,  5.59s/it]


Epoch 1: train_loss=2.2915 train_f1=0.1222 | val_loss=2.1884 val_f1=0.1787
  Saved best model (val_f1=0.1787)


Epoch 2/10 [Val]: 100%|██████████| 7/7 [00:39<00:00,  5.61s/it]


Epoch 2: train_loss=2.1003 train_f1=0.2577 | val_loss=2.0277 val_f1=0.3222
  Saved best model (val_f1=0.3222)


Epoch 3/10 [Val]: 100%|██████████| 7/7 [00:39<00:00,  5.64s/it]


Epoch 3: train_loss=1.9358 train_f1=0.3858 | val_loss=1.8820 val_f1=0.4621
  Saved best model (val_f1=0.4621)


... (10 outputs left out)

Epoch 9/10 [Val]: 100%|██████████| 7/7 [00:40<00:00,  5.79s/it]


Epoch 9: train_loss=1.2695 train_f1=0.7115 | val_loss=1.3163 val_f1=0.6423
  Saved best model (val_f1=0.6423)


Epoch 10/10 [Val]: 100%|██████████| 7/7 [00:40<00:00,  5.76s/it]


Epoch 10: train_loss=1.1978 train_f1=0.7238 | val_loss=1.2569 val_f1=0.6588
  Saved best model (val_f1=0.6588)


In [19]:
wandb.finish()
print(f"\nTraining complete. Best val F1: {best_val_f1:.4f}")

epoch,▁▂▃▃▄▅▆▆▇█
train_acc,▁▂▄▅▆▇▇███
train_f1,▁▃▄▅▆▇▇███
train_loss,█▇▆▅▄▃▃▂▁▁
val_acc,▁▃▅▆▇▇▇███
val_f1,▁▃▅▆▇▇▇▇██
val_loss,█▇▆▅▄▃▃▂▁▁
epoch,10
train_acc,0.72875
train_f1,0.72385
train_loss,1.19781



Training complete. Best val F1: 0.6588
